# Text Analysis System — Customer Reviews

This notebook builds an NLP application that analyses real Yelp customer reviews with three NLP techniques and exposes the results through a FastAPI REST API:

1. **Sentiment analysis** with a pretrained review model (no fine-tuning).
2. **Topic detection** with zero-shot classification.
3. **Summarisation** with a pretrained summarisation model.

## Overview

| Part | Implementation | Section |
|---|---|---|
| Data | Real customer reviews from the public Yelp review dataset | 2–3 |
| Sentiment analysis | Pretrained `nlptown/bert-base-multilingual-uncased-sentiment`, compared with a pretrained Twitter baseline | 4 |
| Topic detection | Zero-shot classification with `facebook/bart-large-mnli` | 5 |
| Summarisation | `facebook/bart-large-cnn` | 6 |
| Interface | FastAPI: `GET /health`, `POST /analyse`, `POST /analyse/batch` | 7–8 |
| Evaluation | 150 reviews for sentiment, 20 separate reviews through the API over HTTP, and manual quality review of topics and summaries | 4–6, 9 |
| Report | `report.md`, generated from the executed results | 10 |

## How to run
1. In Google Colab: **Runtime → Change runtime type → T4 GPU**.
2. **Runtime → Run all.** The first run downloads four models (two of about 1.6 GB).
3. The manual review cells in sections 5 and 6 print the predictions to inspect. Their judgement tables start empty, and the report marks those sections as *pending* until they are filled in and the notebook is re-run from that cell onwards.

## 1. Installation and imports

Colab already includes `transformers`, `torch`, `scikit-learn`, `pandas` and `requests`. Only `datasets`, `fastapi` and `uvicorn` need installing.

In [ ]:
!pip install -q datasets fastapi uvicorn

In [ ]:
import platform
import textwrap
import threading
import time
from typing import Literal

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import requests
import datasets
import transformers
import fastapi
from IPython.display import Markdown, display
from datasets import load_dataset
from transformers import pipeline
from sklearn.metrics import (accuracy_score, precision_recall_fscore_support,
                             classification_report, confusion_matrix, ConfusionMatrixDisplay)

SEED = 42
transformers.set_seed(SEED)
DEVICE = 0 if torch.cuda.is_available() else -1
LABELS = ["negative", "neutral", "positive"]

DATASET_NAME = "Yelp/yelp_review_full"
REVIEW_MODEL = "nlptown/bert-base-multilingual-uncased-sentiment"
BASELINE_MODEL = "cardiffnlp/twitter-roberta-base-sentiment-latest"
TOPIC_MODEL = "facebook/bart-large-mnli"
SUMMARY_MODEL = "facebook/bart-large-cnn"

EVAL_PER_STAR = 30   # 150 evaluation reviews
API_PER_STAR = 4     # 20 API test reviews

VERSIONS = {
    "python": platform.python_version(),
    "torch": torch.__version__,
    "transformers": transformers.__version__,
    "datasets": datasets.__version__,
    "fastapi": fastapi.__version__,
}

pd.set_option("display.max_colwidth", 120)
print("GPU:", torch.cuda.get_device_name(0) if DEVICE == 0 else "not available (running on CPU)")
print("Versions:", VERSIONS)

## 2. Dataset loading and inspection

The data is the public **Yelp Review Full** dataset (`Yelp/yelp_review_full`): real customer reviews, each with the reviewer's own 1–5 star rating.

- `load_dataset()` returns a `DatasetDict` with `train` and `test` splits. Only the **test** split is used: no model is trained here, so the training split is not needed.
- Every review gets a `review_id`, its row position in the original test split. These IDs are used later to keep the evaluation and API test sets completely separate.
- Yelp stores line breaks as the two characters `\n`; they are replaced with spaces.

In [ ]:
yelp = load_dataset(DATASET_NAME)   # returns a DatasetDict
print(yelp)

yelp_test = yelp["test"]
print("\nFeatures:", yelp_test.features)

reviews_all = yelp_test.to_pandas()
# review_id: row position in the test split, used to prevent overlap
reviews_all.insert(0, "review_id", np.arange(len(reviews_all)))
reviews_all["stars"] = reviews_all["label"] + 1   # label 0-4 → 1-5 stars

escaped_line_breaks = reviews_all["text"].str.contains("\\n", regex=False).sum()
reviews_all["text"] = (reviews_all["text"]
                       .str.replace(r"(\\n|\n)+", " ", regex=True)
                       .str.replace(r"\s+", " ", regex=True)
                       .str.strip())
reviews_all["word_count"] = reviews_all["text"].str.split().str.len()

print(f"\nTest split: {len(reviews_all):,} reviews")
print(f"Reviews with escaped line breaks (cleaned): {escaped_line_breaks:,}")
print(f"Empty reviews: {(reviews_all['word_count'] == 0).sum()}")
print(f"Reviews whose text appears more than once: {reviews_all['text'].duplicated(keep=False).sum()}")
print("\nReviews per star rating:")
print(reviews_all["stars"].value_counts().sort_index().to_string())
print("\nWords per review:")
print(reviews_all["word_count"].describe().round(1).to_string())

reviews_all[["review_id", "stars", "word_count", "text"]].head(3)

## 3. Data preparation and sampling

**Mapping star ratings to sentiment.** The reviewer's star rating is the ground-truth label:

| Stars | Sentiment |
|---|---|
| 1–2 | negative |
| 3 | neutral |
| 4–5 | positive |

The same mapping is applied to the review model's predicted stars, so both are compared on the same three classes.

**Sampling.**
- **Evaluation set:** 150 reviews, 30 per star rating, drawn with `random_state=42`. This gives 60 negative, 30 neutral and 60 positive reviews. The classes are not equal in size, so **macro** precision, recall and F1 are reported, which weight each class equally.
- **API test set:** 20 further reviews, 4 per star rating, drawn only from reviews that are *not* in the evaluation set.
- **No overlap:** duplicated texts and empty reviews are removed before sampling, so the same review cannot appear in both sets under two different IDs. The `assert` lines check for overlap in both IDs and texts.

In [ ]:
def stars_to_sentiment(stars):
    """1-2 stars → negative, 3 stars → neutral, 4-5 stars → positive."""
    if stars <= 2:
        return "negative"
    if stars == 3:
        return "neutral"
    return "positive"


reviews_all["true_sentiment"] = reviews_all["stars"].apply(stars_to_sentiment)

# Drop duplicated texts and empty reviews so the two sets cannot overlap
candidates = reviews_all[~reviews_all["text"].duplicated(keep=False) & (reviews_all["word_count"] > 0)]

eval_df = (candidates.groupby("stars")
           .sample(n=EVAL_PER_STAR, random_state=SEED)
           .reset_index(drop=True))

remaining = candidates[~candidates["review_id"].isin(eval_df["review_id"])]
api_df = (remaining.groupby("stars")
          .sample(n=API_PER_STAR, random_state=SEED)
          .reset_index(drop=True))

shared_ids = set(eval_df["review_id"]) & set(api_df["review_id"])
shared_texts = set(eval_df["text"]) & set(api_df["text"])

assert len(eval_df) == 5 * EVAL_PER_STAR and len(api_df) == 5 * API_PER_STAR
assert not shared_ids and not shared_texts, "The evaluation and API test sets overlap"

print(f"Candidates after removing duplicates and empty reviews: {len(candidates):,}")
print(f"Evaluation set: {len(eval_df)} reviews | API test set: {len(api_df)} reviews")
print(f"Shared review IDs: {len(shared_ids)} | Shared texts: {len(shared_texts)}")
print(f"\nEvaluation review IDs (first 10): {sorted(eval_df['review_id'])[:10]}")
print(f"API test review IDs: {sorted(api_df['review_id'])}")

pd.DataFrame({
    "evaluation_set": eval_df["true_sentiment"].value_counts(),
    "api_test_set": api_df["true_sentiment"].value_counts(),
}).reindex(LABELS)

## 4. Sentiment analysis and evaluation

Two **pretrained** models are used without any fine-tuning:

| Model | Trained on | Output |
|---|---|---|
| `nlptown/bert-base-multilingual-uncased-sentiment` (**review model**) | Product reviews | 1–5 stars, mapped to 3 classes with the table in section 3 |
| `cardiffnlp/twitter-roberta-base-sentiment-latest` (**baseline**) | Tweets | negative / neutral / positive directly |

The baseline shows how much it matters that the model was trained on the same kind of text. Inputs longer than 512 tokens are truncated, the maximum both models accept.

In [ ]:
review_model = pipeline("sentiment-analysis", model=REVIEW_MODEL, device=DEVICE)
baseline_model = pipeline("sentiment-analysis", model=BASELINE_MODEL, device=DEVICE)

print("Review model labels:  ", review_model.model.config.id2label)
print("Baseline model labels:", baseline_model.model.config.id2label)


def stars_from_label(label):
    """The review model's labels look like "1 star" or "4 stars"."""
    return int(label.split()[0])


eval_texts = eval_df["text"].tolist()

start = time.time()
review_preds = review_model(eval_texts, batch_size=16, truncation=True, max_length=512)
review_seconds = time.time() - start

start = time.time()
baseline_preds = baseline_model(eval_texts, batch_size=16, truncation=True, max_length=512)
baseline_seconds = time.time() - start

eval_df["pred_stars"] = [stars_from_label(p["label"]) for p in review_preds]
eval_df["confidence"] = [p["score"] for p in review_preds]
eval_df["pred_sentiment"] = eval_df["pred_stars"].apply(stars_to_sentiment)
eval_df["baseline_sentiment"] = [p["label"].lower() for p in baseline_preds]
eval_df["baseline_confidence"] = [p["score"] for p in baseline_preds]

print(f"\nReview model:   {review_seconds:.1f}s for {len(eval_df)} reviews")
print(f"Baseline model: {baseline_seconds:.1f}s for {len(eval_df)} reviews")

eval_df[["review_id", "stars", "true_sentiment", "pred_stars", "pred_sentiment",
         "confidence", "baseline_sentiment"]].head(10)

### Evaluation metrics

For each model: accuracy with an approximate 95% confidence interval, plus macro precision, recall and F1, and a full classification report. With only 150 reviews, each review changes accuracy by about 0.7 percentage points, so the confidence interval shows how far the result could move on a different sample.

In [ ]:
def evaluate(y_true, y_pred):
    precision, recall, f1, _ = precision_recall_fscore_support(
        y_true, y_pred, labels=LABELS, average="macro", zero_division=0)
    accuracy = accuracy_score(y_true, y_pred)
    # 95% confidence interval (normal approximation)
    margin = 1.96 * np.sqrt(accuracy * (1 - accuracy) / len(y_true))
    return {
        "n": len(y_true),
        "accuracy": accuracy,
        "accuracy_low": max(0.0, accuracy - margin),
        "accuracy_high": min(1.0, accuracy + margin),
        "macro_precision": precision,
        "macro_recall": recall,
        "macro_f1": f1,
        "report": classification_report(y_true, y_pred, labels=LABELS, output_dict=True, zero_division=0),
    }


MODEL_COLUMNS = {
    "Review model (nlptown BERT)": "pred_sentiment",
    "Twitter baseline (RoBERTa)": "baseline_sentiment",
}
results = {name: evaluate(eval_df["true_sentiment"], eval_df[column]) for name, column in MODEL_COLUMNS.items()}

comparison = pd.DataFrame({
    name: {
        "Accuracy": f"{r['accuracy']:.1%}",
        "Accuracy 95% CI": f"{r['accuracy_low']:.1%} – {r['accuracy_high']:.1%}",
        "Macro precision": f"{r['macro_precision']:.3f}",
        "Macro recall": f"{r['macro_recall']:.3f}",
        "Macro F1": f"{r['macro_f1']:.3f}",
    }
    for name, r in results.items()
})
display(comparison)

for name, column in MODEL_COLUMNS.items():
    print(f"\n=== {name} ===")
    print(classification_report(eval_df["true_sentiment"], eval_df[column], labels=LABELS, zero_division=0))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

for ax, (name, column) in zip(axes, MODEL_COLUMNS.items()):
    cm = confusion_matrix(eval_df["true_sentiment"], eval_df[column], labels=LABELS)
    ConfusionMatrixDisplay(cm, display_labels=LABELS).plot(ax=ax, cmap="Blues", colorbar=False)
    ax.set_title(f"{name}\naccuracy {results[name]['accuracy']:.1%}")
    ax.set_xlabel("Predicted")
    ax.set_ylabel("True (from star rating)")

plt.tight_layout()
plt.show()

### Error analysis

The review model's mistakes, grouped by type (true → predicted), with five examples.

In [ ]:
errors = eval_df[eval_df["pred_sentiment"] != eval_df["true_sentiment"]]
mistake_counts = (errors["true_sentiment"] + " → " + errors["pred_sentiment"]).value_counts()

print(f"{len(errors)} of {len(eval_df)} reviews misclassified by the review model\n")
print(mistake_counts.to_string(), "\n")

for row in errors.head(5).itertuples():
    print(f"review_id {row.review_id} | {row.stars}★ (true {row.true_sentiment}) "
          f"| predicted {row.pred_stars}★ ({row.pred_sentiment}, confidence {row.confidence:.2f})")
    print(textwrap.fill(row.text[:400] + ("..." if len(row.text) > 400 else ""),
                        width=110, initial_indent="  ", subsequent_indent="  "))
    print()

### Limitations of the model comparison
- **Different training text:** the baseline was trained on tweets and the review model on product reviews. The comparison measures how well each *transfers* to Yelp reviews, not which architecture is better.
- **Different output formats:** the review model predicts 1–5 stars, which our mapping turns into 3 classes; the baseline predicts 3 classes directly. The choice of mapping (for example, treating 3 stars as neutral) affects the review model's scores.
- **"Neutral" means different things:** a 3-star Yelp review is often mixed (praise *and* complaints), while the baseline's "neutral" means factual or emotionless text.
- **Small sample:** 150 reviews from one random seed. Differences inside the confidence intervals are not conclusive.
- **Possible training overlap:** it cannot be ruled out that either pretrained model saw Yelp reviews during training.

## 5. Topic detection and quality assessment

**Zero-shot classification** (`facebook/bart-large-mnli`) checks whether each review is about each candidate topic, without any topic-labelled training data.
- `multi_label=True` scores every topic independently, so one review can have several topics.
- A topic counts as detected when its score is at least **0.5**.
- All six scores are kept so near-misses can be inspected.

Yelp has no topic labels, so quality is judged in two ways:
1. **Keyword agreement:** a rough, automatic check, used only as a supplement.
2. **Manual inspection** of 20 predictions, recorded in the review table below.

In [ ]:
TOPICS = ["food quality", "customer service", "price", "atmosphere", "cleanliness", "waiting time"]
TOPIC_THRESHOLD = 0.5

topic_model = pipeline("zero-shot-classification", model=TOPIC_MODEL, device=DEVICE)

start = time.time()
topic_results = topic_model(eval_texts, candidate_labels=TOPICS, multi_label=True, batch_size=8)
topic_seconds = time.time() - start

eval_df["topic_scores"] = [dict(zip(r["labels"], r["scores"])) for r in topic_results]
eval_df["topics"] = [[t for t in TOPICS if scores[t] >= TOPIC_THRESHOLD] for scores in eval_df["topic_scores"]]

reviews_with_topic = (eval_df["topics"].str.len() > 0).mean()
print(f"Topic detection: {topic_seconds:.1f}s for {len(eval_df)} reviews")
print(f"Reviews with at least one topic: {reviews_with_topic:.1%}")
print(f"Average topics per review: {eval_df['topics'].str.len().mean():.2f}\n")

topic_counts = (eval_df.explode("topics").dropna(subset=["topics"])
                .groupby(["topics", "true_sentiment"]).size()
                .unstack(fill_value=0)
                .reindex(index=TOPICS, columns=LABELS, fill_value=0))
topic_share = topic_counts.div(eval_df["true_sentiment"].value_counts().reindex(LABELS), axis=1) * 100
print("% of reviews in each sentiment group that mention each topic:")
print(topic_share.round(1).to_string())

topic_share.plot(kind="barh", figsize=(9, 5), color=["#d9534f", "#f0ad4e", "#5cb85c"])
plt.xlabel("% of reviews in the sentiment group")
plt.title("Detected topics by true sentiment")
plt.tight_layout()
plt.show()

### Keyword agreement (rough supplementary check)

For each topic, a short keyword list approximates "the review mentions this topic".
- **Precision vs keywords:** of the reviews tagged with a topic, the share that contain one of its keywords.
- **Recall vs keywords:** of the reviews containing a keyword, the share tagged with the topic.

Keywords miss synonyms and match unrelated uses (for example "fast" in "breakfast" is avoided, but "quick" can describe anything), so these numbers are only indicative.

In [ ]:
TOPIC_KEYWORDS = {
    "food quality":     ["food", "dish", "dishes", "taste", "tasty", "delicious", "meal", "menu", "flavor", "flavour", "bland"],
    "customer service": ["service", "staff", "waiter", "waitress", "server", "rude", "friendly", "manager", "helpful"],
    "price":            ["price", "prices", "expensive", "cheap", "cost", "overpriced", "worth", "dollars", "pricey"],
    "atmosphere":       ["atmosphere", "ambiance", "ambience", "decor", "music", "vibe", "cozy", "loud"],
    "cleanliness":      ["clean", "dirty", "filthy", "hygiene", "bathroom", "bathrooms", "sticky"],
    "waiting time":     ["wait", "waited", "waiting", "slow", "minutes", "hour", "hours", "quick"],
}

lower_texts = eval_df["text"].str.lower()
keyword_rows = []

for topic, words in TOPIC_KEYWORDS.items():
    has_keyword = lower_texts.str.contains(r"\b(?:" + "|".join(words) + r")\b", regex=True)
    detected = eval_df["topics"].apply(lambda topics: topic in topics)
    both = (has_keyword & detected).sum()
    keyword_rows.append({
        "topic": topic,
        "detected": int(detected.sum()),
        "keyword_present": int(has_keyword.sum()),
        "precision_vs_keywords": both / detected.sum() if detected.sum() else np.nan,
        "recall_vs_keywords": both / has_keyword.sum() if has_keyword.sum() else np.nan,
    })

keyword_check = pd.DataFrame(keyword_rows)
mean_keyword_precision = keyword_check["precision_vs_keywords"].mean()
mean_keyword_recall = keyword_check["recall_vs_keywords"].mean()

display(keyword_check.style.format({"precision_vs_keywords": "{:.0%}", "recall_vs_keywords": "{:.0%}"}))
print(f"Mean precision vs keywords: {mean_keyword_precision:.0%}")
print(f"Mean recall vs keywords:    {mean_keyword_recall:.0%}")

### Manual inspection of 20 topic predictions

The 20 reviews below are sampled from the evaluation set with `random_state=42`. For each one, the detected topics and all six scores are printed next to the full review text.

Each prediction is judged and recorded in `TOPIC_REVIEW` in the next cell:
- **correct:** the detected topics match what the review actually discusses, and nothing important is missing.
- **partly correct:** at least one topic is right, but a topic is missing or wrongly added.
- **incorrect:** the detected topics do not reflect the review.

In [ ]:
TOPIC_REVIEW_SAMPLE = eval_df.sample(n=20, random_state=SEED)

for row in TOPIC_REVIEW_SAMPLE.itertuples():
    scores = ", ".join(f"{topic} {score:.2f}" for topic, score in row.topic_scores.items())
    print(f"review_id {row.review_id} | {row.stars}★ | detected: {row.topics}")
    print(f"  scores: {scores}")
    print(textwrap.fill(row.text, width=110, initial_indent="  ", subsequent_indent="  "))
    print()

In [ ]:
# Filled in from the 20 predictions printed above.
# Format: review_id: {"verdict": "correct" | "partly correct" | "incorrect", "note": "..."}
TOPIC_REVIEW = {
}

TOPIC_OBSERVATIONS = [
]

topic_sample_ids = set(TOPIC_REVIEW_SAMPLE["review_id"])
topic_reviewed = {rid: v for rid, v in TOPIC_REVIEW.items() if rid in topic_sample_ids}
topic_review_complete = len(topic_reviewed) == len(topic_sample_ids)

if topic_review_complete:
    topic_verdicts = pd.Series([v["verdict"] for v in topic_reviewed.values()]).value_counts()
    print(f"Manual topic review of {len(topic_reviewed)} predictions:")
    print(topic_verdicts.to_string())
    print("\nObservations:")
    for observation in TOPIC_OBSERVATIONS:
        print("-", observation)
else:
    topic_verdicts = pd.Series(dtype=int)
    print(f"Manual topic review pending: {len(topic_reviewed)} of {len(topic_sample_ids)} predictions judged.")

## 6. Summarisation and quality assessment

`facebook/bart-large-cnn` writes a 15–60 token summary. Only reviews of **120+ words** are summarised, because shorter reviews are already brief.

The quality of 20 summaries is checked in three ways:
1. **Compression:** summary length as a share of the original.
2. **Sentiment preservation:** the review model is run on each summary. Does it give the same sentiment as for the full review?
3. **Manual inspection** of 10 summaries for factual accuracy, keeping the important information, and invented details.

The summariser accepts at most 1,024 tokens, so longer reviews are truncated. The number of truncated reviews is reported.

In [ ]:
MIN_WORDS_TO_SUMMARISE = 120
SUMMARY_SAMPLE_SIZE = 20

summariser = pipeline("summarization", model=SUMMARY_MODEL, device=DEVICE)

long_candidates = eval_df[eval_df["word_count"] >= MIN_WORDS_TO_SUMMARISE]
print(f"Evaluation reviews with {MIN_WORDS_TO_SUMMARISE}+ words: {len(long_candidates)}")
long_reviews = long_candidates.sample(n=min(SUMMARY_SAMPLE_SIZE, len(long_candidates)), random_state=SEED).copy()

summary_max_input = summariser.tokenizer.model_max_length
long_reviews["input_tokens"] = [len(summariser.tokenizer(text)["input_ids"]) for text in long_reviews["text"]]
truncated_reviews = int((long_reviews["input_tokens"] > summary_max_input).sum())

start = time.time()
summaries = summariser(long_reviews["text"].tolist(), max_length=60, min_length=15,
                       do_sample=False, truncation=True, batch_size=4)
summary_seconds = time.time() - start

long_reviews["summary"] = [s["summary_text"] for s in summaries]
long_reviews["summary_words"] = long_reviews["summary"].str.split().str.len()
compression = (long_reviews["summary_words"] / long_reviews["word_count"]).mean()

summary_preds = review_model(long_reviews["summary"].tolist(), truncation=True, max_length=512)
long_reviews["summary_sentiment"] = [stars_to_sentiment(stars_from_label(p["label"])) for p in summary_preds]
sentiment_preserved = (long_reviews["summary_sentiment"] == long_reviews["pred_sentiment"]).mean()

print(f"Summarised {len(long_reviews)} reviews in {summary_seconds:.1f}s")
print(f"Reviews truncated to {summary_max_input} tokens: {truncated_reviews}")
print(f"Average length: {long_reviews['word_count'].mean():.0f} → {long_reviews['summary_words'].mean():.0f} words "
      f"(summary is {compression:.0%} of the original)")
print(f"Summary sentiment = full-review prediction: {sentiment_preserved:.0%}")

### Manual inspection of 10 summaries

The first 10 summarised reviews are printed in full with their summaries. Each is judged and recorded in `SUMMARY_REVIEW` in the next cell:
- **factually_accurate:** does the summary only state things the review says? (`True` / `False`)
- **key_points_kept:** are the review's main points kept? (`"yes"` / `"partly"` / `"no"`)
- **invented_details:** does the summary add anything not in the review? (`True` / `False`)

In [ ]:
SUMMARY_REVIEW_SAMPLE = long_reviews.head(10)

for row in SUMMARY_REVIEW_SAMPLE.itertuples():
    print(f"review_id {row.review_id} | {row.stars}★ | {row.word_count} words → {row.summary_words} words")
    print(textwrap.fill(row.text, width=110, initial_indent="  REVIEW:  ", subsequent_indent="           "))
    print(textwrap.fill(row.summary, width=110, initial_indent="  SUMMARY: ", subsequent_indent="           "))
    print()

In [ ]:
# Filled in from the 10 summaries printed above.
# Format: review_id: {"factually_accurate": True/False, "key_points_kept": "yes" | "partly" | "no",
#                     "invented_details": True/False, "note": "..."}
SUMMARY_REVIEW = {
}

SUMMARY_OBSERVATIONS = [
]

summary_sample_ids = set(SUMMARY_REVIEW_SAMPLE["review_id"])
summary_reviewed = {rid: v for rid, v in SUMMARY_REVIEW.items() if rid in summary_sample_ids}
summary_review_complete = len(summary_reviewed) == len(summary_sample_ids)

if summary_review_complete:
    summary_judgements = pd.DataFrame(summary_reviewed.values())
    accurate_count = int(summary_judgements["factually_accurate"].sum())
    invented_count = int(summary_judgements["invented_details"].sum())
    key_points_counts = summary_judgements["key_points_kept"].value_counts()
    print(f"Factually accurate: {accurate_count} of {len(summary_judgements)}")
    print(f"Invented details:   {invented_count} of {len(summary_judgements)}")
    print("Key points kept:")
    print(key_points_counts.to_string())
    print("\nObservations:")
    for observation in SUMMARY_OBSERVATIONS:
        print("-", observation)
else:
    print(f"Manual summary review pending: {len(summary_reviewed)} of {len(summary_sample_ids)} summaries judged.")

## 7. Integrated analysis engine

`ReviewAnalyser` combines the three techniques behind one method, so the API only has to call `analyse()` or `analyse_batch()`. It reuses the models already loaded above instead of loading them a second time.

In [ ]:
class ReviewAnalyser:
    def __init__(self, sentiment_model, topic_model, summariser):
        self.sentiment_model = sentiment_model
        self.topic_model = topic_model
        self.summariser = summariser

    def analyse_batch(self, texts):
        sentiments = self.sentiment_model(texts, truncation=True, max_length=512)
        topic_results = self.topic_model(texts, candidate_labels=TOPICS, multi_label=True)
        # Some versions return a dict for a single input.
        if isinstance(topic_results, dict):
            topic_results = [topic_results]

        results = []
        for text, sentiment, topic_result in zip(texts, sentiments, topic_results):
            stars = stars_from_label(sentiment["label"])
            word_count = len(text.split())

            summary = None
            if word_count >= MIN_WORDS_TO_SUMMARISE:
                summary = self.summariser(text, max_length=60, min_length=15,
                                          do_sample=False, truncation=True)[0]["summary_text"]

            results.append({
                "sentiment": stars_to_sentiment(stars),
                "stars": stars,
                "confidence": round(float(sentiment["score"]), 4),
                "topics": [label for label, score in zip(topic_result["labels"], topic_result["scores"])
                           if score >= TOPIC_THRESHOLD],
                "summary": summary,
                "word_count": word_count,
            })
        return results

    def analyse(self, text):
        return self.analyse_batch([text])[0]


analyser = ReviewAnalyser(review_model, topic_model, summariser)
analyser.analyse("The food was delicious but we waited an hour and the staff were rude.")

## 8. FastAPI implementation

| Endpoint | Request | Response |
|---|---|---|
| `GET /health` | — | status and the model names |
| `POST /analyse` | `{"text": "..."}` | one `ReviewAnalysis` |
| `POST /analyse/batch` | `{"texts": ["...", ...]}` (1–50 texts) | `{"count": n, "results": [ReviewAnalysis, ...]}` |

**Validation:** Pydantic rejects missing fields, empty or whitespace-only text, texts over 10,000 characters, and batches of 0 or more than 50 texts. FastAPI returns these as **422** responses that explain what was wrong.

**Error handling:** an unexpected failure inside the models returns a **500** response with a short message, instead of the server crashing.

**Response models:** declaring `response_model` makes FastAPI check every response against the declared fields before sending it.

In [ ]:
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel, Field, field_validator

MAX_TEXT_CHARS = 10_000
MAX_BATCH_SIZE = 50


class ReviewRequest(BaseModel):
    text: str = Field(..., min_length=1, max_length=MAX_TEXT_CHARS)

    @field_validator("text")
    @classmethod
    def text_not_blank(cls, value):
        if not value.strip():
            raise ValueError("text must not be blank")
        return value.strip()


class BatchRequest(BaseModel):
    texts: list[str] = Field(..., min_length=1, max_length=MAX_BATCH_SIZE)

    @field_validator("texts")
    @classmethod
    def texts_valid(cls, values):
        cleaned = [value.strip() for value in values]
        for i, value in enumerate(cleaned):
            if not value:
                raise ValueError(f"texts[{i}] must not be blank")
            if len(value) > MAX_TEXT_CHARS:
                raise ValueError(f"texts[{i}] is longer than {MAX_TEXT_CHARS} characters")
        return cleaned


class ReviewAnalysis(BaseModel):
    sentiment: Literal["negative", "neutral", "positive"]
    stars: int = Field(..., ge=1, le=5)
    confidence: float = Field(..., ge=0, le=1)
    topics: list[str]
    summary: str | None
    word_count: int


class BatchAnalysis(BaseModel):
    count: int
    results: list[ReviewAnalysis]


class HealthResponse(BaseModel):
    status: str
    models: dict[str, str]


app = FastAPI(title="Review Text Analysis API")


@app.get("/health", response_model=HealthResponse)
def health():
    return {"status": "ok",
            "models": {"sentiment": REVIEW_MODEL, "topics": TOPIC_MODEL, "summary": SUMMARY_MODEL}}


@app.post("/analyse", response_model=ReviewAnalysis)
def analyse_review(request: ReviewRequest):
    try:
        return analyser.analyse(request.text)
    except Exception as error:
        raise HTTPException(status_code=500, detail=f"Analysis failed: {type(error).__name__}: {error}")


@app.post("/analyse/batch", response_model=BatchAnalysis)
def analyse_reviews(request: BatchRequest):
    try:
        results = analyser.analyse_batch(request.texts)
    except Exception as error:
        raise HTTPException(status_code=500, detail=f"Analysis failed: {type(error).__name__}: {error}")
    return {"count": len(results), "results": results}


print("Routes:", [route.path for route in app.routes if route.path in ("/health", "/analyse", "/analyse/batch")])

## 9. API testing

The API runs as a real **uvicorn** server in a background thread, and every test below is a real HTTP request sent with `requests`.
1. **Endpoint and validation tests:** a valid request, plus invalid ones that should be rejected with 422.
2. **20 separate real reviews:** the API test set from section 3, which shares no review with the evaluation set, sent to `POST /analyse/batch`. The returned sentiment is compared with each reviewer's star rating.

In [ ]:
import uvicorn

API_HOST, API_PORT = "127.0.0.1", 8000
API_URL = f"http://{API_HOST}:{API_PORT}"

if "server" in globals() and server.started:
    print("API already running at", API_URL)
else:
    server = uvicorn.Server(uvicorn.Config(app, host=API_HOST, port=API_PORT,
                                           log_level="warning", loop="asyncio"))
    server_thread = threading.Thread(target=server.run, daemon=True)
    server_thread.start()

    for _ in range(60):
        if server.started:
            break
        time.sleep(0.5)

    if not server.started:
        raise RuntimeError("The API server did not start")
    print("API running at", API_URL)

In [ ]:
endpoint_tests = [
    # (description, method, path, JSON body, expected status)
    ("Health check", "GET", "/health", None, 200),
    ("Analyse one real review", "POST", "/analyse", {"text": api_df["text"].iloc[0]}, 200),
    ("Empty text", "POST", "/analyse", {"text": ""}, 422),
    ("Whitespace-only text", "POST", "/analyse", {"text": "    "}, 422),
    ("Missing text field", "POST", "/analyse", {}, 422),
    ("Empty batch", "POST", "/analyse/batch", {"texts": []}, 422),
    ("Batch of 51 texts", "POST", "/analyse/batch", {"texts": ["Good food."] * 51}, 422),
    ("Batch with a blank text", "POST", "/analyse/batch", {"texts": ["Good food.", " "]}, 422),
]

test_rows = []
for description, method, path, body, expected in endpoint_tests:
    response = requests.request(method, f"{API_URL}{path}", json=body, timeout=300)
    if response.status_code == 422:
        detail = response.json()["detail"][0]["msg"]
    else:
        detail = str(response.json())[:80]
    test_rows.append({"test": description, "request": f"{method} {path}", "expected": expected,
                      "status": response.status_code, "passed": response.status_code == expected,
                      "response": detail})

endpoint_results = pd.DataFrame(test_rows)
endpoint_tests_passed = bool(endpoint_results["passed"].all())
display(endpoint_results)
print(f"Endpoint tests passed: {endpoint_results['passed'].sum()} of {len(endpoint_results)}")

single = requests.post(f"{API_URL}/analyse", json={"text": api_df["text"].iloc[0]}, timeout=300).json()
print("\nExample POST /analyse response:")
print(single)

In [ ]:
start = time.time()
response = requests.post(f"{API_URL}/analyse/batch", json={"texts": api_df["text"].tolist()}, timeout=900)
api_seconds = time.time() - start

print(f"POST /analyse/batch → status {response.status_code} in {api_seconds:.1f}s")
response.raise_for_status()
batch = response.json()
assert batch["count"] == len(api_df), "The API returned a different number of results"

api_df["api_sentiment"] = [r["sentiment"] for r in batch["results"]]
api_df["api_stars"] = [r["stars"] for r in batch["results"]]
api_df["api_confidence"] = [r["confidence"] for r in batch["results"]]
api_df["api_topics"] = [r["topics"] for r in batch["results"]]
api_df["api_summary"] = [r["summary"] for r in batch["results"]]

api_eval = evaluate(api_df["true_sentiment"], api_df["api_sentiment"])
api_seconds_per_review = api_seconds / len(api_df)
api_summaries = int(api_df["api_summary"].notna().sum())

print(f"\nAPI sentiment accuracy on {api_eval['n']} separate real reviews: {api_eval['accuracy']:.0%} "
      f"(95% CI {api_eval['accuracy_low']:.0%} – {api_eval['accuracy_high']:.0%})")
print(f"Macro precision {api_eval['macro_precision']:.3f} | macro recall {api_eval['macro_recall']:.3f} "
      f"| macro F1 {api_eval['macro_f1']:.3f}")
print(f"Time per review (sentiment + topics + summary): {api_seconds_per_review:.2f}s")
print(f"Reviews summarised (120+ words): {api_summaries} of {len(api_df)}\n")
print(pd.crosstab(api_df["true_sentiment"], api_df["api_sentiment"], rownames=["true"], colnames=["API"])
      .reindex(index=LABELS, columns=LABELS, fill_value=0).to_string())

api_df[["review_id", "stars", "true_sentiment", "api_stars", "api_sentiment", "api_confidence", "api_topics", "text"]]

## 10. Findings and limitations

The next cell writes `report.md` **only from values computed above**. Sections that depend on the manual reviews say *pending* until the review tables in sections 5 and 6 are filled in.

The last cell is a final check: it confirms that each part of the system ran, and that the key numbers in `report.md` match the notebook. It shows **TODO** for the manual reviews until `TOPIC_REVIEW` and `SUMMARY_REVIEW` are filled in and the notebook is re-run from section 5.

In [ ]:
main = results["Review model (nlptown BERT)"]
baseline = results["Twitter baseline (RoBERTa)"]

recalls = {label: main["report"][label]["recall"] for label in LABELS}
hardest_class = min(recalls, key=recalls.get)
most_common_mistake = mistake_counts.index[0] if len(mistake_counts) else "none"
top_negative_topic = topic_share["negative"].idxmax()
top_positive_topic = topic_share["positive"].idxmax()

intervals_overlap = not (main["accuracy_low"] > baseline["accuracy_high"]
                         or baseline["accuracy_low"] > main["accuracy_high"])
higher_model = "review model" if main["macro_f1"] >= baseline["macro_f1"] else "Twitter baseline"
comparison_note = ("The accuracy confidence intervals overlap, so the difference is not conclusive on 150 reviews."
                   if intervals_overlap else
                   "The accuracy confidence intervals do not overlap.")

# The audit cell checks these exact strings appear in report.md.
KEY_VALUES = {
    "review_accuracy": f"{main['accuracy']:.1%}",
    "review_macro_f1": f"{main['macro_f1']:.3f}",
    "baseline_accuracy": f"{baseline['accuracy']:.1%}",
    "baseline_macro_f1": f"{baseline['macro_f1']:.3f}",
    "api_accuracy": f"{api_eval['accuracy']:.0%}",
}

if topic_review_complete:
    topic_manual_text = (
        f"{len(topic_reviewed)} predictions judged: "
        + ", ".join(f"{count} {verdict}" for verdict, count in topic_verdicts.items()) + "."
        + "".join(f"\n  - {observation}" for observation in TOPIC_OBSERVATIONS)
    )
else:
    topic_manual_text = f"Pending: {len(topic_reviewed)} of {len(topic_sample_ids)} predictions judged."

if summary_review_complete:
    summary_manual_text = (
        f"{len(summary_judgements)} summaries judged: {accurate_count} factually accurate, "
        f"{invented_count} with invented details; key points kept: "
        + ", ".join(f"{verdict} {count}" for verdict, count in key_points_counts.items()) + "."
        + "".join(f"\n  - {observation}" for observation in SUMMARY_OBSERVATIONS)
    )
else:
    summary_manual_text = f"Pending: {len(summary_reviewed)} of {len(summary_sample_ids)} summaries judged."

report = f"""# Text Analysis System — Report

## Architecture

```
Review text
   │
   ▼
ReviewAnalyser ──► Sentiment: {REVIEW_MODEL}
               │     (1–5 stars → negative / neutral / positive)
               ──► Topics:    zero-shot {TOPIC_MODEL}
               │     ({len(TOPICS)} topics, threshold {TOPIC_THRESHOLD})
               ──► Summary:   {SUMMARY_MODEL}
                     (reviews of {MIN_WORDS_TO_SUMMARISE}+ words only)
   │
   ▼
FastAPI (uvicorn): GET /health · POST /analyse · POST /analyse/batch → JSON
```

All models are pretrained and used without fine-tuning. Pydantic validates requests and responses;
invalid input returns 422 and model failures return 500.

## Reproducibility

| Item | Value |
|---|---|
| Dataset | `{DATASET_NAME}`, test split ({len(reviews_all):,} reviews) |
| Ground truth | Reviewer star rating: 1–2 negative, 3 neutral, 4–5 positive |
| Random seed | {SEED} |
| Evaluation set | {len(eval_df)} reviews, {EVAL_PER_STAR} per star rating |
| API test set | {len(api_df)} reviews, {API_PER_STAR} per star rating, no shared IDs or texts with the evaluation set |
| Sentiment model / baseline | `{REVIEW_MODEL}` / `{BASELINE_MODEL}` |
| Topic model | `{TOPIC_MODEL}` |
| Summary model | `{SUMMARY_MODEL}` |
| Hardware | {torch.cuda.get_device_name(0) if DEVICE == 0 else "CPU"} |
| Library versions | {", ".join(f"{k} {v}" for k, v in VERSIONS.items())} |

## Results

### Sentiment ({len(eval_df)} evaluation reviews)

| Metric | Review model | Twitter baseline |
|---|---|---|
| Accuracy | {KEY_VALUES['review_accuracy']} | {KEY_VALUES['baseline_accuracy']} |
| Accuracy 95% CI | {main['accuracy_low']:.1%} – {main['accuracy_high']:.1%} | {baseline['accuracy_low']:.1%} – {baseline['accuracy_high']:.1%} |
| Macro precision | {main['macro_precision']:.3f} | {baseline['macro_precision']:.3f} |
| Macro recall | {main['macro_recall']:.3f} | {baseline['macro_recall']:.3f} |
| Macro F1 | {KEY_VALUES['review_macro_f1']} | {KEY_VALUES['baseline_macro_f1']} |
| Recall: negative / neutral / positive | {recalls['negative']:.0%} / {recalls['neutral']:.0%} / {recalls['positive']:.0%} | {baseline['report']['negative']['recall']:.0%} / {baseline['report']['neutral']['recall']:.0%} / {baseline['report']['positive']['recall']:.0%} |

### Topic detection

- Reviews with at least one topic: {reviews_with_topic:.1%}
- Keyword agreement (rough check): precision {mean_keyword_precision:.0%}, recall {mean_keyword_recall:.0%}
- Manual review of 20 predictions: {topic_manual_text}

### Summarisation ({len(long_reviews)} reviews of {MIN_WORDS_TO_SUMMARISE}+ words)

- Summary length: {compression:.0%} of the original on average ({truncated_reviews} reviews truncated to {summary_max_input} tokens)
- Summary sentiment matches the full-review prediction: {sentiment_preserved:.0%}
- Manual review of 10 summaries: {summary_manual_text}

### API test ({len(api_df)} separate real reviews over HTTP)

- Endpoint and validation tests passed: {int(endpoint_results['passed'].sum())} of {len(endpoint_results)}
- Sentiment accuracy: {KEY_VALUES['api_accuracy']} (95% CI {api_eval['accuracy_low']:.0%} – {api_eval['accuracy_high']:.0%}), macro F1 {api_eval['macro_f1']:.3f}
- Time per review (all three techniques): {api_seconds_per_review:.2f}s

## Key findings

- The **{higher_model}** has the higher macro F1 ({max(main['macro_f1'], baseline['macro_f1']):.3f} vs {min(main['macro_f1'], baseline['macro_f1']):.3f}). {comparison_note}
- **{hardest_class.capitalize()}** is the review model's hardest class (recall {recalls[hardest_class]:.0%}). Its most common mistake is **{most_common_mistake}**.
- The topic most often detected in negative reviews is **{top_negative_topic}**; in positive reviews it is **{top_positive_topic}**.
- Summaries average {compression:.0%} of the original length, and their sentiment matches the full review {sentiment_preserved:.0%} of the time.

## Limitations

- Star ratings are a noisy label: a reviewer's rating does not always match the tone of their text.
- The model comparison mixes a 5-star review model (mapped to 3 classes) with a 3-class tweet model; the mapping and the different meaning of "neutral" affect the result.
- 150 evaluation reviews and 20 API reviews from one random seed: small samples, so all metrics carry real uncertainty (see the confidence intervals).
- Topic detection has no labelled ground truth; keyword agreement is rough, and the manual review covers only 20 predictions.
- Summaries were checked automatically for length and sentiment, and manually on only 10 examples.
- The API runs inside a notebook for demonstration; it is not set up for production use (no authentication, rate limiting or monitoring).
"""

with open("report.md", "w", encoding="utf-8") as f:
    f.write(report)

display(Markdown(report))
print("Saved report.md")

In [ ]:
with open("report.md", encoding="utf-8") as f:
    saved_report = f.read()

checks = [
    ("Yelp reviews loaded", DATASET_NAME == "Yelp/yelp_review_full"),
    ("Sentiment analysis ran", "pred_sentiment" in eval_df.columns),
    ("Topic detection ran", "topics" in eval_df.columns),
    ("Summarisation ran", "summary" in long_reviews.columns),
    ("All API endpoint tests passed", endpoint_tests_passed),
    ("150 reviews evaluated", len(eval_df) == 150),
    ("20 separate reviews analysed over HTTP", len(api_df) == 20 and batch["count"] == 20),
    ("No overlap between evaluation and API sets", not shared_ids and not shared_texts),
    ("Manual topic review complete", topic_review_complete),
    ("Manual summary review complete", summary_review_complete),
    ("report.md matches the notebook's key metrics",
     all(value in saved_report for value in KEY_VALUES.values())),
]

for check, passed in checks:
    print(f"[{'PASS' if passed else 'TODO'}] {check}")

print(f"\n{sum(passed for _, passed in checks)} of {len(checks)} checks passed")